# 3. Label Encoding & Stratified Train–Validation Split

**Goal**: Convert human-readable labels into stable numerical IDs and create a fair 80-20 split that respects class imbalance.

In [ ]:
import pandas as pd
import json
from sklearn.model_selection import train_test_split
import os

print("Libraries imported.")

## 2.1 Freeze the Label Space

In [ ]:
file_path = 'round1/Primary_Emotions_Processed.xlsx'
df = pd.read_excel(file_path)

# Extract unique labels and sort alphabetically
unique_labels = sorted(df['Primary'].unique())
print(f"Total unique labels: {len(unique_labels)}")

# Create mapping
label_map = {label: idx for idx, label in enumerate(unique_labels)}

# Save mapping immediately
map_path = 'round1/label_map.json'
with open(map_path, 'w') as f:
    json.dump(label_map, f, indent=4)

print(f"Label map saved to {map_path}")
print("Sample mapping:")
print({k: label_map[k] for k in list(label_map)[:5]})

## 2.2 Encode Labels

In [ ]:
df['label_id'] = df['Primary'].map(label_map)
display(df[['Primary', 'label_id']].head())

## 2.3 Verify Label Coverage

In [ ]:
missing = df['label_id'].isnull().sum()
print(f"Rows with missing label_id: {missing}")

unique_ids = df['label_id'].nunique()
print(f"Total unique label_ids: {unique_ids}")

if missing == 0 and unique_ids == 46:
    print("✅ Label encoding verified.")
else:
    print("❌ ERROR: Label encoding failed!")

## 2.4 Stratified Train–Validation Split

In [ ]:
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['label_id'], 
    random_state=42
)

print(f"Train shape: {train_df.shape}")
print(f"Val shape:   {val_df.shape}")

## 2.5 Validate the Split

In [ ]:
# Total Integrity Check
total_samples = len(train_df) + len(val_df)
expected_samples = len(df)
print(f"Total Samples (Train + Val): {total_samples} / {expected_samples}")

# Label Distribution Check
train_labels = set(train_df['label_id'].unique())
val_labels = set(val_df['label_id'].unique())
all_labels = set(df['label_id'].unique())

missing_in_train = all_labels - train_labels
missing_in_val = all_labels - val_labels

print(f"Labels missing in Train: {missing_in_train}")
print(f"Labels missing in Val:   {missing_in_val}")

if len(missing_in_train) == 0 and len(missing_in_val) == 0:
    print("✅ Stratification successful. All labels present in both sets.")
else:
    print("⚠️ Warning: Some labels missing in split (likely extremely rare classes).")

## 2.6 Save Artifacts

In [ ]:
train_path = 'round1/train.xlsx'
val_path = 'round1/val.xlsx'

train_df.to_excel(train_path, index=False)
val_df.to_excel(val_path, index=False)

print(f"Train set saved to {train_path}")
print(f"Validation set saved to {val_path}")
print("Step 2 Complete ✅")